<a href="https://colab.research.google.com/github/fernandascarvalho/datascience/blob/main/Projeto_Data_Science%20/%20Av_Projeto_CRISPDM_Amazon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto CRISP-DM — Amazon: notas baixas em caixas de assinatura
**Problema de negócio:** o que leva um cliente de assinatura a dar nota baixa (1 ou 2 estrelas) e onde a
marca deve agir para reduzir o risco de cancelamento e a má reputação?

**Dados REAIS e públicos:** *Amazon Reviews 2023* (McAuley Lab, UCSD), categoria `Subscription_Boxes`
(avaliações reais de clientes da Amazon dos EUA, 2017–2023) e o arquivo de metadados dos produtos.
Não são dados publicados pela Amazon e não são do Brasil. Confira a licença de uso antes de citar.

**Como rodar (Google Colab):** execute a célula de download, depois as demais em ordem.
Estrutura: Fase 1 (negócio) → Fase 2 (dados) → Fase 3 (preparação) → Fase 4 (modelagem) → Fase 5 (avaliação).

In [ ]:
# Baixa os dois arquivos (se ainda não estiverem na pasta 'dados/'). Precisa de internet.
from pathlib import Path
import urllib.request

BASE_URL = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw"
ARQS = {
    "reviews_Subscription_Boxes.jsonl": f"{BASE_URL}/review_categories/Subscription_Boxes.jsonl",
    "meta_Subscription_Boxes.jsonl": f"{BASE_URL}/meta_categories/meta_Subscription_Boxes.jsonl",
}
Path("dados").mkdir(exist_ok=True)
for nome, url in ARQS.items():
    destino = Path("dados") / nome
    if not destino.exists():
        urllib.request.urlretrieve(url, destino)
    print(nome, round(destino.stat().st_size / 1e6, 1), "MB")

reviews_Subscription_Boxes.jsonl 9.0 MB
meta_Subscription_Boxes.jsonl 1.4 MB


In [ ]:
import warnings
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score, accuracy_score, roc_curve, silhouette_score
import statsmodels.api as sm
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize, proportions_ztest

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170, "display.max_columns", 30, "display.max_colwidth", 60)

SEED = 42
PASTA_GRAF = Path("graficos"); PASTA_GRAF.mkdir(exist_ok=True)
PASTA_RES = Path("resultados"); PASTA_RES.mkdir(exist_ok=True)
RES = {}


def br(x, nd=1):
    return f"{x:,.{nd}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def pct(x, nd=0):
    return br(100 * x, nd) + "%"


def fmt_p(p):
    return "< 0,001" if p < 0.001 else br(p, 3)


# ---- estilo dos gráficos ----
AZUL, LARANJA, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
AZUL_CLARO, AZUL_ESCURO = "#86b6ef", "#104281"
TINTA, TINTA2, MUDO = "#0b0b0b", "#52514e", "#898781"
GRADE, EIXO, SUP = "#e1e0d9", "#c3c2b7", "#fcfcfb"
NOTA_REAL = "Dados reais: Amazon Reviews 2023 (McAuley Lab, UCSD), categoria Subscription Boxes"


def novo_grafico(largura=7.5, altura=4.2, ncols=1, **kw):
    fig, axs = plt.subplots(1, ncols, figsize=(largura, altura), dpi=130, facecolor=SUP, **kw)
    lista = np.atleast_1d(axs)
    for ax in lista:
        ax.set_facecolor(SUP)
        for s in ("top", "right"):
            ax.spines[s].set_visible(False)
        for s in ("left", "bottom"):
            ax.spines[s].set_color(EIXO)
        ax.tick_params(colors=TINTA2, labelsize=9)
        ax.grid(axis="y", color=GRADE, linewidth=0.8)
        ax.set_axisbelow(True)
    return fig, lista


def titulo(fig, texto, subtitulo=None):
    fig.text(0.01, 0.985, texto, fontsize=11.5, fontweight="bold", color=TINTA, ha="left", va="top")
    if subtitulo:
        fig.text(0.01, 0.925, subtitulo, fontsize=9, color=TINTA2, ha="left", va="top")


def salvar(fig, nome, nota=NOTA_REAL):
    fig.text(0.01, 0.012, nota, fontsize=7.5, color=MUDO, ha="left", va="bottom")
    fig.savefig(PASTA_GRAF / nome, facecolor=SUP)
    plt.show()
    plt.close(fig)

## Fase 1 — Compreensão do Negócio
**Negócio:** marcas de caixas de assinatura vendidas na Amazon. Nota baixa derruba a conversão de novos
clientes (CAC mais alto) e sinaliza risco de cancelamento (LTV menor).

**Pergunta:** quais motivos aparecem nas avaliações de 1 e 2 estrelas e que ação de marketing reduz o dano?

**Variável-alvo:** `nota_baixa = 1` se a nota é 1 ou 2.

**Critérios de sucesso:** (1) identificar os temas que mais se associam a nota baixa, com teste estatístico;
(2) modelo que classifique texto de cliente insatisfeito com recall ≥ 80% (para triagem de atendimento);
(3) retorno financeiro da ação maior que 1 nos cenários de premissa.

**Premissas de negócio (NÃO vêm dos dados; são suposições do grupo, editáveis):**
o arquivo de produtos não traz preço, então ticket, margem, custos e CAC abaixo são ilustrativos, em US$.

In [ ]:
TICKET_MENSAL = 35.0        # PREMISSA (US$/mês): os dados não trazem preço
MARGEM = 0.30               # PREMISSA: margem de contribuição
MESES_EXTRA_RETIDO = 6      # PREMISSA: meses adicionais de assinatura de quem é retido
P_CANCELA_SE_INSATISFEITO = 0.50   # PREMISSA: parcela dos insatisfeitos que cancelaria sem ação
CUSTO_CONTATO = 2.0         # PREMISSA (US$/contato): resposta ativa por e-mail/mensagem
CUSTO_COM_INCENTIVO = 6.0   # PREMISSA (US$/contato): resposta + incentivo (ex.: desconto)
CENARIOS_RETENCAO = [0.05, 0.10, 0.20]  # PREMISSA: % dos que cancelariam e são retidos pela ação
CAC = 45.0                  # PREMISSA (US$): custo de aquisição de um assinante
CHURN_MENSAL = 0.10         # PREMISSA: cancelamento mensal médio
META_RECALL = 0.80
VALOR_RETIDO = TICKET_MENSAL * MARGEM * MESES_EXTRA_RETIDO
print(f"Valor de um assinante retido (margem × meses extras): US$ {br(VALOR_RETIDO, 2)}")

Valor de um assinante retido (margem × meses extras): US$ 63,00


## Fase 2 — Compreensão dos Dados
### 2.1 Coleta e primeira olhada

In [ ]:
rev = pd.read_json("dados/reviews_Subscription_Boxes.jsonl", lines=True)
meta = pd.read_json("dados/meta_Subscription_Boxes.jsonl", lines=True)
print(f"Avaliações: {rev.shape} | Produtos: {meta.shape}")
display(rev.drop(columns=["images"]).head(3))
print("Preço preenchido nos metadados:", pct(pd.to_numeric(meta.price, errors="coerce").notna().mean()))

Avaliações: (16216, 10) | Produtos: (641, 14)


,rating,title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,1,USELESS,Absolutely useless nonsense and a complete waste of mone...,B07G584SHG,B09WC47S3V,AEMJ2EG5ODOCYUTI54NBXZHDJGSQ,2020-10-08 05:10:57.705,2,True
1,2,Manufactured where?,"With a couple of the items, I wasn't quite sure of their...",B07QL1JRCN,B07QL1JRCN,AEEJBFZKUBEEMBZUZJV4UHFVEEBQ,2020-12-27 23:12:15.433,20,True
2,1,Little bang for your buck.,Two SMALL stuffed animals and 2 little bags of treats fo...,B07RBYJN37,B08N5QKX1Y,AGSVZNZBTSGQBKZDZTQYEZHGDPCQ,2021-01-06 12:48:35.319,4,True


Preço preenchido nos metadados: 0%


### 2.2 Qualidade dos dados

In [ ]:
rev["ano"] = pd.to_datetime(rev.timestamp, unit="ms").dt.year
rev["data"] = pd.to_datetime(rev.timestamp, unit="ms")
qual = {
    "Avaliações no arquivo": len(rev),
    "Duplicadas (mesmo cliente, produto e instante)": int(rev.duplicated(["user_id", "parent_asin", "timestamp"]).sum()),
    "Duplicadas (mesmo cliente, produto e texto)": int(rev.duplicated(["user_id", "parent_asin", "text"]).sum()),
    "Texto vazio": int((rev.text.str.strip() == "").sum()),
    "Valores ausentes em nota/texto/data": int(rev[["rating", "text", "timestamp"]].isna().sum().sum()),
    "Avaliações sem produto nos metadados": int((~rev.parent_asin.isin(meta.parent_asin)).sum()),
    "Produtos sem preço nos metadados": int(pd.to_numeric(meta.price, errors="coerce").isna().sum()),
}
display(pd.Series(qual, name="quantidade").to_frame())
print("Período:", rev.data.min().date(), "a", rev.data.max().date(), "(2023 é ano parcial)")
RES["qualidade"] = qual
RES["periodo"] = [str(rev.data.min().date()), str(rev.data.max().date())]

,quantidade
Avaliações no arquivo,16216
"Duplicadas (mesmo cliente, produto e instante)",200
"Duplicadas (mesmo cliente, produto e texto)",208
Texto vazio,11
Valores ausentes em nota/texto/data,0
Avaliações sem produto nos metadados,0
Produtos sem preço nos metadados,641


Período: 2017-01-24 a 2023-08-30 (2023 é ano parcial)


### 2.3 Exploração preliminar

In [ ]:
rev["nota_baixa"] = (rev.rating <= 2).astype(int)
dist = rev.rating.value_counts(normalize=True).sort_index()
RES["dist_notas"] = {str(int(k)): float(v) for k, v in dist.items()}
RES["taxa_nota_baixa_bruta"] = float(rev.nota_baixa.mean())
print("Distribuição das notas:", {int(k): pct(v, 1) for k, v in dist.items()})
print("Nota baixa (1 ou 2):", pct(rev.nota_baixa.mean(), 1))

fig, (ax,) = novo_grafico(7.2, 4.0)
cores = [LARANJA if k <= 2 else AZUL for k in dist.index]
ax.bar(dist.index.astype(int), 100 * dist.values, width=0.6, color=cores)
for k, v in dist.items():
    ax.text(int(k), 100 * v + 1.2, pct(v), ha="center", fontsize=10, fontweight="bold", color=TINTA)
ax.set_xlabel("Nota (estrelas)", color=TINTA2); ax.set_ylabel("% das avaliações", color=TINTA2)
ax.set_ylim(0, 100 * dist.max() + 10)
titulo(fig, f"{pct(dist.get(5.0, 0))} das avaliações são nota 5, mas {pct(rev.nota_baixa.mean())} são nota 1 ou 2",
       f"{br(len(rev), 0)} avaliações de caixas de assinatura (laranja = nota baixa)")
fig.subplots_adjust(top=0.82, bottom=0.16)
salvar(fig, "g1_distribuicao_notas.png")

Distribuição das notas: {1: '15,9%', 2: '9,2%', 3: '10,8%', 4: '10,3%', 5: '53,9%'}
Nota baixa (1 ou 2): 25,1%


## Fase 3 — Preparação dos Dados
**Limpeza → Transformação (variáveis novas) → Formatação.**

In [ ]:
log = [("Avaliações no arquivo original", len(rev))]
d = rev.copy()
n0 = len(d); d = d.drop_duplicates(["user_id", "parent_asin", "timestamp"]); log.append(("Removidas duplicatas (cliente + produto + instante)", n0 - len(d)))
n0 = len(d); d = d.drop_duplicates(["user_id", "parent_asin", "text"]); log.append(("Removidas duplicatas (cliente + produto + texto)", n0 - len(d)))
d["texto_completo"] = (d.title.fillna("") + ". " + d.text.fillna("")).str.strip()
n0 = len(d); d = d[d.text.str.strip() != ""]; log.append(("Removidas avaliações sem texto", n0 - len(d)))
log.append(("Avaliações finais para modelagem", len(d)))
display(pd.DataFrame(log, columns=["Etapa de limpeza", "Linhas"]))
RES["log_limpeza"] = dict(log)
d = d.reset_index(drop=True)

,Etapa de limpeza,Linhas
0,Avaliações no arquivo original,16216
1,Removidas duplicatas (cliente + produto + instante),200
2,Removidas duplicatas (cliente + produto + texto),8
3,Removidas avaliações sem texto,11
4,Avaliações finais para modelagem,15997


In [ ]:
# ---- Variáveis novas (feature engineering) ----
TOPICOS = {
    "cobranca_cancelamento": r"\b(charg(e|ed|es|ing)|bill(ed|ing)?|cancel(l?ed|ing|lation)?|refund(ed)?|auto[- ]?renew\w*|unsubscrib\w*|credit card)\b",
    "preco_valor": r"\b(price[sd]?|pricey|expensive|overpriced|costs?|costly|waste of (money|time)|not worth|worth|value|rip[- ]?off|\$\s?\d+)\b",
    "qualidade_defeito": r"\b(quality|poor|flimsy|junk|broke|broken|defective|cheaply made|garbage|trash|crap|crappy)\b",
    "entrega_embalagem": r"\b(ship(ped|ping)?|deliver(y|ed)?|arriv(e|ed|al)|late|delay(ed)?|damag(e|ed)|packag(e|ing)|never (got|received)|didn'?t (get|receive)|missing)\b",
    "atendimento": r"\b(customer (service|support)|support|contact(ed)?|respon(d|ded|se)|e-?mail(ed)?|called|rude|seller)\b",
    "expectativa_decepcao": r"\b(disappoint\w*|expect(ed|ations?)?|misleading|false advertising|not as (described|advertised|pictured)|smaller|tiny|small|same (items?|things?)|repeat(ed)?|boring)\b",
    "presente": r"\b(gift|gifted|birthday|christmas|present)\b",
}
NOMES_TOPICOS = {
    "cobranca_cancelamento": "Cobrança e cancelamento", "preco_valor": "Preço e valor",
    "qualidade_defeito": "Qualidade e defeito", "entrega_embalagem": "Entrega e embalagem",
    "atendimento": "Atendimento", "expectativa_decepcao": "Expectativa e decepção", "presente": "Presente",
}
txt = d.texto_completo.str.lower()
for k, rx in TOPICOS.items():
    d[k] = txt.str.contains(rx, regex=True).astype(int)

TIPOS = [
    ("Plantas", r"plant|succulent|cact|flower|bonsai|garden|bouquet|herb"),
    ("Pets", r"\bcat\b|\bdog\b|\bpet\b|kitten|puppy|catlady"),
    ("Vestuário", r"t-shirt|\btee\b|sock|clothing|apparel|shirt|hat\b"),
    ("Comida e bebida", r"snack|coffee|\btea\b|candy|chocolate|food|wine|beer|spice|jerky|cookie|bean|sauce|cocktail|meal|fruit|nut"),
    ("Geek e colecionáveis", r"funko|loot|geek|gamer|gaming|anime|collectible|comic|star wars|marvel|crate|toy|expanse|pop!|movie|dvd|dungeon"),
    ("Beleza e bem-estar", r"beauty|makeup|skin|soap|bath|hair|nail|candle|wellness|self[- ]care|spa\b"),
    ("Artesanato e papelaria", r"book|craft|paper|card|art\b|sticker|puzzle|knit|crochet|diy|kit|sewing|journal"),
]


def tipo_produto(titulo_):
    t = str(titulo_).lower()
    for nome, rx in TIPOS:
        if re.search(rx, t):
            return nome
    return "Outros"


meta["tipo_produto"] = meta.title.map(tipo_produto)
d = d.merge(meta[["parent_asin", "tipo_produto", "rating_number", "title"]].rename(columns={"title": "produto"}), on="parent_asin", how="left")
d["n_palavras"] = d.text.str.split().str.len()
d["log_palavras"] = np.log1p(d.n_palavras)
d["log_popularidade"] = np.log1p(d.rating_number.fillna(d.rating_number.median()))
d["n_avaliacoes_cliente"] = d.groupby("user_id").user_id.transform("size")
d["log_n_cliente"] = np.log1p(d.n_avaliacoes_cliente)
d["tem_foto"] = (d.images.str.len() > 0).astype(int)
d["compra_verificada"] = d.verified_purchase.astype(int)
d["exclamacoes"] = d.text.str.count("!").clip(upper=5)
d["ano_num"] = d.ano.astype(int)
print("Produtos por tipo (nos metadados):", meta.tipo_produto.value_counts().to_dict())
print("Avaliações por tipo:", d.tipo_produto.value_counts().to_dict())
print("Menções por tema:", {NOMES_TOPICOS[k]: pct(d[k].mean(), 1) for k in TOPICOS})
RES["freq_topicos"] = {k: float(d[k].mean()) for k in TOPICOS}
RES["tipos_n"] = d.tipo_produto.value_counts().to_dict()

Produtos por tipo (nos metadados): {'Outros': 186, 'Geek e colecionáveis': 101, 'Comida e bebida': 89, 'Vestuário': 81, 'Artesanato e papelaria': 70, 'Beleza e bem-estar': 45, 'Plantas': 35, 'Pets': 34}
Avaliações por tipo: {'Geek e colecionáveis': 3905, 'Beleza e bem-estar': 3301, 'Outros': 2458, 'Pets': 2007, 'Comida e bebida': 1508, 'Vestuário': 1389, 'Plantas': 833, 'Artesanato e papelaria': 596}
Menções por tema: {'Cobrança e cancelamento': '10,6%', 'Preço e valor': '29,6%', 'Qualidade e defeito': '15,4%', 'Entrega e embalagem': '14,5%', 'Atendimento': '4,2%', 'Expectativa e decepção': '17,9%', 'Presente': '10,2%'}


### 2.4 Hipóteses analíticas (H1 a H6) — testes estatísticos
As hipóteses e as direções esperadas foram definidas **antes** dos testes. Nível de significância de 5%.

In [ ]:
base = d.nota_baixa.mean()
RES["taxa_nota_baixa"] = float(base); RES["n_final"] = len(d)


def teste_tema(col):
    com, sem = d[d[col] == 1].nota_baixa, d[d[col] == 0].nota_baixa
    ct = pd.crosstab(d[col], d.nota_baixa)
    p = stats.chi2_contingency(ct)[1]
    return com.mean(), sem.mean(), len(com), p


linhas = []
for h, col, afirm in (("H1", "cobranca_cancelamento", "Quem fala de cobrança/cancelamento dá mais nota baixa"),
                      ("H2", "expectativa_decepcao", "Quem fala de expectativa/decepção dá mais nota baixa"),
                      ("H3", "qualidade_defeito", "Quem fala de qualidade/defeito dá mais nota baixa")):
    com, sem, n_com, p = teste_tema(col)
    linhas.append((h, afirm, "Qui-quadrado", f"{pct(com, 1)} (cita, n={n_com}) vs {pct(sem, 1)} (não cita)", p, (p < 0.05) and (com > sem)))

ver = d.groupby("compra_verificada").nota_baixa.mean()
p4 = stats.chi2_contingency(pd.crosstab(d.compra_verificada, d.nota_baixa))[1]
linhas.append(("H4", "Compra não verificada tem mais nota baixa", "Qui-quadrado",
               f"{pct(ver[0], 1)} (não verificada) vs {pct(ver[1], 1)} (verificada)", p4, (p4 < 0.05) and (ver[0] > ver[1])))

a, b = d.loc[d.nota_baixa == 1, "helpful_vote"], d.loc[d.nota_baixa == 0, "helpful_vote"]
p5 = stats.mannwhitneyu(a, b, alternative="greater").pvalue
linhas.append(("H5", "Avaliações de nota baixa recebem mais votos úteis", "Mann-Whitney",
               f"mediana {br(a.median(), 0)} (baixa) vs {br(b.median(), 0)} (alta); média {br(a.mean())} vs {br(b.mean())}", p5, p5 < 0.05))

ct6 = pd.crosstab(d.tipo_produto, d.nota_baixa)
p6 = stats.chi2_contingency(ct6)[1]
t6 = d.groupby("tipo_produto").nota_baixa.mean()
linhas.append(("H6", "A taxa de nota baixa varia conforme o tipo de produto", "Qui-quadrado",
               f"de {pct(t6.min(), 1)} ({t6.idxmin()}) a {pct(t6.max(), 1)} ({t6.idxmax()})", p6, p6 < 0.05))

tab_h = pd.DataFrame(linhas, columns=["Hipótese", "Afirmação", "Teste", "Resultado", "p-valor", "Confirmada?"])
tab_h["p-valor"] = tab_h["p-valor"].map(fmt_p)
tab_h["Confirmada?"] = tab_h["Confirmada?"].map({True: "Sim", False: "Não"})
display(tab_h)
tab_h.to_csv(PASTA_RES / "tab_hipoteses.csv", index=False)
RES["hipoteses"] = tab_h.to_dict(orient="records")

# tendência por ano
por_ano = d.groupby("ano").nota_baixa.agg(["mean", "size"])
display(por_ano.assign(mean=lambda x: (100 * x["mean"]).round(1)))
RES["nota_baixa_por_ano"] = {int(k): {"taxa": float(v["mean"]), "n": int(v["size"])} for k, v in por_ano.iterrows()}
p_ano = stats.chi2_contingency(pd.crosstab(d.ano, d.nota_baixa))[1]
RES["p_ano"] = float(p_ano)

,Hipótese,Afirmação,Teste,Resultado,p-valor,Confirmada?
0,H1,Quem fala de cobrança/cancelamento dá mais nota baixa,Qui-quadrado,"60,0% (cita, n=1699) vs 20,9% (não cita)","< 0,001",Sim
1,H2,Quem fala de expectativa/decepção dá mais nota baixa,Qui-quadrado,"50,4% (cita, n=2858) vs 19,6% (não cita)","< 0,001",Sim
2,H3,Quem fala de qualidade/defeito dá mais nota baixa,Qui-quadrado,"36,6% (cita, n=2468) vs 23,0% (não cita)","< 0,001",Sim
3,H4,Compra não verificada tem mais nota baixa,Qui-quadrado,"39,4% (não verificada) vs 23,1% (verificada)","< 0,001",Sim
4,H5,Avaliações de nota baixa recebem mais votos úteis,Mann-Whitney,"mediana 6 (baixa) vs 1 (alta); média 11,8 vs 3,8","< 0,001",Sim
5,H6,A taxa de nota baixa varia conforme o tipo de produto,Qui-quadrado,"de 20,2% (Plantas) a 28,4% (Outros)","< 0,001",Sim


,mean,size
ano,,
2017,16.6,710
2018,28.9,1903
2019,20.9,3047
2020,24.4,4209
2021,28.2,3640
2022,26.3,2033
2023,26.8,455


In [ ]:
fig, (ax,) = novo_grafico(7.8, 4.2)
anos = por_ano.index.to_numpy()
ax.plot(anos, 100 * por_ano["mean"], color=AZUL, linewidth=2.2, marker="o", markersize=6)
ax.axhline(100 * base, color=MUDO, linewidth=1, linestyle="--")
ax.text(anos[-1], 100 * base + 0.8, f"média {pct(base, 1)}", color=TINTA2, fontsize=8.5, ha="right")
for x, (r, n) in zip(anos, zip(por_ano["mean"], por_ano["size"])):
    ax.text(x, 100 * r + 1.5, f"{pct(r)}\n(n={br(n, 0)})", ha="center", fontsize=8.5, color=TINTA)
ax.set_ylim(0, 100 * por_ano["mean"].max() + 12)
ax.set_xticks(anos); ax.set_xticklabels([f"{a}*" if a == anos[-1] else str(a) for a in anos])
ax.set_ylabel("% de avaliações com nota 1 ou 2", color=TINTA2)
tendencia = "varia entre os anos" if p_ano < 0.05 else "não muda de forma significativa entre os anos"
titulo(fig, f"A taxa de nota baixa {tendencia} (de {pct(por_ano['mean'].min())} a {pct(por_ano['mean'].max())})",
       f"Teste qui-quadrado entre anos: p {'< 0,001' if p_ano < 0.001 else '= ' + br(p_ano, 3)}. *2023 só até agosto")
fig.subplots_adjust(top=0.82, bottom=0.14)
salvar(fig, "g2_nota_baixa_por_ano.png")

In [ ]:
linhas = []
for k in TOPICOS:
    com, sem, n_com, p = teste_tema(k)
    linhas.append({"tema": NOMES_TOPICOS[k], "col": k, "taxa_com": com, "taxa_sem": sem, "n_com": n_com, "p": p})
tab_top = pd.DataFrame(linhas).sort_values("taxa_com")
tab_top.to_csv(PASTA_RES / "tab_temas.csv", index=False)
RES["temas"] = tab_top.to_dict(orient="records")

fig, (ax,) = novo_grafico(8.2, 4.6)
ax.grid(axis="x", color=GRADE, linewidth=0.8); ax.grid(axis="y", visible=False)
cor = [LARANJA if r > base else AZUL for r in tab_top.taxa_com]
ax.barh(range(len(tab_top)), 100 * tab_top.taxa_com, height=0.55, color=cor)
ax.axvline(100 * base, color=MUDO, linewidth=1, linestyle="--")
ax.text(100 * base + 0.6, len(tab_top) - 0.45, f"média geral {pct(base, 1)}", color=TINTA2, fontsize=8.5, va="bottom")
for i, r in enumerate(tab_top.itertuples()):
    ax.text(100 * r.taxa_com + 0.8, i, f"{pct(r.taxa_com, 1)}  (n={br(r.n_com, 0)})", va="center", fontsize=8.5, color=TINTA)
ax.set_yticks(range(len(tab_top))); ax.set_yticklabels(tab_top.tema, fontsize=9)
ax.set_xlim(0, 100 * tab_top.taxa_com.max() + 18); ax.set_ylim(-0.6, len(tab_top) - 0.1 + 0.5)
ax.set_xlabel("% com nota baixa entre as avaliações que citam o tema", color=TINTA2)
topo = tab_top.iloc[-1]
titulo(fig, f"Quem cita {topo.tema.lower()} dá nota baixa em {pct(topo.taxa_com)} das vezes, contra {pct(base)} no geral",
       "Temas identificados por palavras-chave no texto da avaliação (laranja = acima da média)")
fig.subplots_adjust(top=0.82, bottom=0.14, left=0.22)
salvar(fig, "g3_nota_baixa_por_tema.png")

### Divisão treino/teste e formatação
Estratificada, 70/30. O vetorizador de texto e o padronizador aprendem **só no treino** (dentro dos pipelines).

In [ ]:
NUM = ["log_palavras", "log_popularidade", "log_n_cliente", "ano_num", "exclamacoes"]
CAT = ["tipo_produto"]
BIN = ["compra_verificada", "tem_foto"] + list(TOPICOS)
FEAT = NUM + CAT + BIN
treino, teste = train_test_split(d, test_size=0.30, stratify=d.nota_baixa, random_state=SEED)
Xtr, ytr, Xte, yte = treino[FEAT + ["texto_completo"]], treino.nota_baixa.to_numpy(), teste[FEAT + ["texto_completo"]], teste.nota_baixa.to_numpy()
print(f"Treino: {len(treino)} ({pct(ytr.mean(), 1)} nota baixa) | Teste: {len(teste)} ({pct(yte.mean(), 1)} nota baixa)")
RES["n_treino"], RES["n_teste"] = len(treino), len(teste)


def prep_estrutura():
    return ColumnTransformer([
        ("num", StandardScaler(), NUM),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), CAT),
        ("bin", "passthrough", BIN),
    ], sparse_threshold=0)

Treino: 11197 (25,1% nota baixa) | Teste: 4800 (25,1% nota baixa)


## Fase 4 — Modelagem
### 4.1 Comparação de algoritmos (validação cruzada em 5 partes, métrica AUC)
- **A. Atributos + temas:** só o que é conhecido sobre produto, cliente e temas citados (sem a nota).
- **B. Texto completo (TF-IDF):** o texto da avaliação vira números (palavras e pares de palavras).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
FEAT_ONLY = FEAT


def pipe_estrutura(modelo):
    return Pipeline([("prep", prep_estrutura()), ("clf", modelo)])


buscas = {
    "A. Regressão logística (atributos + temas)": (GridSearchCV(
        pipe_estrutura(LogisticRegression(solver="liblinear", max_iter=2000)),
        {"clf__C": [0.1, 1, 10]}, cv=cv, scoring="roc_auc"), FEAT_ONLY),
    "A. Árvore de decisão (atributos + temas)": (GridSearchCV(
        pipe_estrutura(DecisionTreeClassifier(random_state=SEED)),
        {"clf__max_depth": [3, 4, 5], "clf__min_samples_leaf": [20, 50]}, cv=cv, scoring="roc_auc"), FEAT_ONLY),
    "A. Floresta aleatória (atributos + temas)": (GridSearchCV(
        pipe_estrutura(RandomForestClassifier(n_estimators=250, random_state=SEED, n_jobs=-1)),
        {"clf__max_depth": [5, None], "clf__min_samples_leaf": [5, 20]}, cv=cv, scoring="roc_auc"), FEAT_ONLY),
    "B. Regressão logística (texto TF-IDF)": (GridSearchCV(
        Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=5, stop_words="english", sublinear_tf=True)),
                  ("clf", LogisticRegression(solver="liblinear", max_iter=2000))]),
        {"clf__C": [0.3, 1, 3, 10]}, cv=cv, scoring="roc_auc"), "texto_completo"),
}

linhas, probs = [], {}
for nome, (busca, cols) in buscas.items():
    busca.fit(Xtr[cols], ytr)
    p = busca.predict_proba(Xte[cols])[:, 1]
    probs[nome] = p
    pred = (p >= 0.5).astype(int)
    linhas.append({"Modelo": nome, "AUC (validação cruzada)": busca.best_score_, "AUC (teste)": roc_auc_score(yte, p),
                   "Recall (limiar 0,5)": recall_score(yte, pred), "Precisão (limiar 0,5)": precision_score(yte, pred),
                   "F1 (limiar 0,5)": f1_score(yte, pred), "Acurácia": accuracy_score(yte, pred),
                   "Melhores parâmetros": str({k.split("__")[1]: v for k, v in busca.best_params_.items()})})
tab_modelos = pd.DataFrame(linhas)
display(tab_modelos.round(3))
base_acc = max(yte.mean(), 1 - yte.mean())
print(f"Referência (sempre prever 'nota alta'): acurácia {pct(base_acc, 1)}, AUC 0,50")
tab_modelos.to_csv(PASTA_RES / "tab_modelos.csv", index=False)
RES["modelos"] = tab_modelos.to_dict(orient="records"); RES["baseline_acuracia"] = float(base_acc)

NOME_A = "A. Regressão logística (atributos + temas)"
NOME_B = "B. Regressão logística (texto TF-IDF)"
auc_A, auc_B = roc_auc_score(yte, probs[NOME_A]), roc_auc_score(yte, probs[NOME_B])
RES["auc_A"], RES["auc_B"] = float(auc_A), float(auc_B)

,Modelo,AUC (validação cruzada),AUC (teste),"Recall (limiar 0,5)","Precisão (limiar 0,5)","F1 (limiar 0,5)",Acurácia,Melhores parâmetros
0,A. Regressão logística (atributos + temas),0.776,0.775,0.293,0.628,0.400,0.779,{'C': 0.1}
1,A. Árvore de decisão (atributos + temas),0.764,0.770,0.314,0.613,0.415,0.778,"{'max_depth': 5, 'min_samples_leaf': 20}"
2,A. Floresta aleatória (atributos + temas),0.794,0.797,0.355,0.628,0.453,0.785,"{'max_depth': None, 'min_samples_leaf': 5}"
3,B. Regressão logística (texto TF-IDF),0.931,0.934,0.588,0.808,0.681,0.862,{'C': 1}


Referência (sempre prever 'nota alta'): acurácia 74,9%, AUC 0,50


In [ ]:
fig, (ax,) = novo_grafico(6.4, 5.4)
ax.grid(axis="x", color=GRADE, linewidth=0.8)
cores = {0: AZUL_CLARO, 1: MUDO, 2: AQUA, 3: LARANJA}
for i, (nome, p) in enumerate(probs.items()):
    fpr, tpr, _ = roc_curve(yte, p)
    ax.plot(fpr, tpr, color=cores[i], linewidth=2.4 if nome == NOME_B else 1.5, label=f"{nome.split('(')[0].strip()} · {nome.split('(')[1][:-1]} (AUC {br(roc_auc_score(yte, p), 2)})")
ax.plot([0, 1], [0, 1], color=MUDO, linewidth=1, linestyle="--", label="Acaso (AUC 0,50)")
ax.set_xlabel("Falsos positivos (taxa)", color=TINTA2); ax.set_ylabel("Notas baixas identificadas (recall)", color=TINTA2)
ax.legend(frameon=False, fontsize=7.5, loc="lower right", labelcolor=TINTA2)
if auc_B > auc_A + 0.1:
    tit = f"O texto prevê a nota baixa (AUC {br(auc_B, 2)}); os atributos, bem menos (AUC {br(auc_A, 2)})"
else:
    tit = f"Texto (AUC {br(auc_B, 2)}) e atributos (AUC {br(auc_A, 2)}) têm desempenho parecido"
titulo(fig, tit, "Curva ROC no conjunto de teste")
fig.subplots_adjust(top=0.85, bottom=0.14)
salvar(fig, "g4_roc_modelos.png")

### 4.2 O que explica a nota baixa? Razões de chances (modelo A, sem regularização)
OR > 1 aumenta a chance de nota baixa; OR < 1 reduz. Variáveis numéricas padronizadas: OR por +1 desvio-padrão.

In [ ]:
prep_sm = prep_estrutura()
Xm = prep_sm.fit_transform(Xtr[FEAT])
nomes = [n.split("__", 1)[1] for n in prep_sm.get_feature_names_out()]
logit = sm.Logit(ytr, sm.add_constant(pd.DataFrame(Xm, columns=nomes))).fit(disp=0)
ic = logit.conf_int()
tab_or = pd.DataFrame({"variavel": nomes, "OR": np.exp(logit.params[nomes].values),
                       "IC95_inf": np.exp(ic.loc[nomes, 0].values), "IC95_sup": np.exp(ic.loc[nomes, 1].values), "p": logit.pvalues[nomes].values})
ROT = {"log_palavras": "Tamanho do texto (+1 DP)", "log_popularidade": "Popularidade do produto (+1 DP)",
       "log_n_cliente": "Nº de avaliações do cliente (+1 DP)", "ano_num": "Ano da avaliação (+1 DP)", "exclamacoes": "Exclamações (+1 DP)",
       "compra_verificada": "Compra verificada", "tem_foto": "Tem foto"}
ROT.update({k: f"Cita: {NOMES_TOPICOS[k].lower()}" for k in TOPICOS})


def rotulo(v):
    if v in ROT:
        return ROT[v]
    if v.startswith("tipo_produto_"):
        return f"Tipo: {v.split('_', 2)[2]} (vs. {sorted(d.tipo_produto.unique())[0]})"
    return v


tab_or["Variável"] = tab_or.variavel.map(rotulo)
tab_or["significativa"] = tab_or.p < 0.05
tab_or = tab_or.sort_values("OR", ascending=False)
display(tab_or[["Variável", "OR", "IC95_inf", "IC95_sup", "p", "significativa"]].round(3))
tab_or.to_csv(PASTA_RES / "tab_razoes_de_chances.csv", index=False)
RES["or"] = tab_or[["Variável", "OR", "IC95_inf", "IC95_sup", "p", "significativa"]].to_dict(orient="records")

,Variável,OR,IC95_inf,IC95_sup,p,significativa
14,Cita: cobrança e cancelamento,4.455,3.859,5.144,0.000,True
19,Cita: expectativa e decepção,3.764,3.352,4.227,0.000,True
15,Cita: preço e valor,1.928,1.738,2.138,0.000,True
18,Cita: atendimento,1.740,1.383,2.188,0.000,True
16,Cita: qualidade e defeito,1.517,1.335,1.723,0.000,True
17,Cita: entrega e embalagem,1.151,1.001,1.323,0.048,True
9,Tipo: Pets (vs. Artesanato e papelaria),1.096,0.817,1.471,0.540,False
6,Tipo: Comida e bebida (vs. Artesanato e papelaria),1.040,0.776,1.394,0.792,False
11,Tipo: Vestuário (vs. Artesanato e papelaria),1.018,0.756,1.372,0.906,False
3,Ano da avaliação (+1 DP),0.995,0.940,1.054,0.872,False


In [ ]:
sig = tab_or[tab_or.significativa].sort_values("OR")
fig, (ax,) = novo_grafico(8.6, 0.42 * len(sig) + 2.0)
ax.grid(axis="x", color=GRADE, linewidth=0.8); ax.grid(axis="y", visible=False)
ax.barh(range(len(sig)), np.log(sig.OR), height=0.55, color=[LARANJA if o > 1 else AZUL for o in sig.OR])
ax.axvline(0, color=EIXO, linewidth=1)
for i, (o, lo, hi) in enumerate(zip(sig.OR, sig.IC95_inf, sig.IC95_sup)):
    ax.plot([np.log(lo), np.log(hi)], [i, i], color=TINTA2, linewidth=1)
    ax.text(np.log(hi) + 0.05 if o > 1 else np.log(lo) - 0.05, i, f"OR {br(o, 2)}", va="center", ha="left" if o > 1 else "right", fontsize=8.5, color=TINTA)
ax.set_yticks(range(len(sig))); ax.set_yticklabels(sig["Variável"], fontsize=8.5)
lim = max(abs(np.log(sig.IC95_inf)).max(), abs(np.log(sig.IC95_sup)).max()) + 0.6
ax.set_xlim(-lim, lim)
ticks = [t for t in (0.25, 0.5, 1, 2, 4, 8) if -lim < np.log(t) < lim]
ax.set_xticks(np.log(ticks)); ax.set_xticklabels([br(t, 2).rstrip("0").rstrip(",") for t in ticks])
ax.set_xlabel("Razão de chances de nota baixa (escala logarítmica; barra fina = IC 95%)", color=TINTA2)
ax.text(0.98, 1.02, "aumenta a chance", color=LARANJA, fontsize=9, fontweight="bold", ha="right", transform=ax.transAxes)
ax.text(0.02, 1.02, "reduz a chance", color=AZUL, fontsize=9, fontweight="bold", ha="left", transform=ax.transAxes)
titulo(fig, f"{len(sig)} fatores se destacam, mantendo os demais iguais", "Só variáveis estatisticamente significativas (p < 0,05), modelo A no treino")
fig.subplots_adjust(top=0.80, bottom=0.17, left=0.34)
salvar(fig, "g5_razoes_de_chances.png")

### 4.3 Palavras que distinguem notas baixas e altas (modelo B, texto)

In [ ]:
mb = buscas[NOME_B][0].best_estimator_
vocab = np.array(mb.named_steps["tfidf"].get_feature_names_out())
coef = mb.named_steps["clf"].coef_[0]
ordem = np.argsort(coef)
pal_baixa = list(zip(vocab[ordem[::-1][:15]], coef[ordem[::-1][:15]]))
pal_alta = list(zip(vocab[ordem[:15]], coef[ordem[:15]]))
display(pd.DataFrame({"palavras de nota baixa": [f"{w} ({c:.2f})" for w, c in pal_baixa], "palavras de nota alta": [f"{w} ({c:.2f})" for w, c in pal_alta]}))
RES["palavras_baixa"] = [(w, float(c)) for w, c in pal_baixa]; RES["palavras_alta"] = [(w, float(c)) for w, c in pal_alta]

fig, axs = novo_grafico(9.0, 5.2, ncols=2, sharex=False)
for ax, lista, cor, rot in ((axs[0], pal_baixa[:12][::-1], LARANJA, "Mais associadas a nota baixa"), (axs[1], pal_alta[:12][::-1], AZUL, "Mais associadas a nota alta")):
    ax.grid(axis="x", color=GRADE, linewidth=0.8); ax.grid(axis="y", visible=False)
    ax.barh(range(len(lista)), [abs(c) for _, c in lista], height=0.6, color=cor)
    ax.set_yticks(range(len(lista))); ax.set_yticklabels([w for w, _ in lista], fontsize=9)
    ax.set_xlabel(rot, color=TINTA2, fontsize=9)
titulo(fig, "As palavras que mais separam nota baixa de nota alta", "Peso das palavras e pares de palavras no modelo B (regressão logística sobre TF-IDF)")
fig.subplots_adjust(top=0.84, bottom=0.14, left=0.13, right=0.98, wspace=0.28)
salvar(fig, "g6_palavras.png")

,palavras de nota baixa,palavras de nota alta
0,disappointed (4.77),love (-7.89)
1,cheap (4.08),great (-6.89)
2,cancelled (3.62),loved (-6.63)
3,disappointing (3.53),loves (-4.21)
4,poor (3.19),fun (-4.14)
5,waste (3.10),nice (-3.57)
6,terrible (2.97),awesome (-3.56)
7,junk (2.81),amazing (-3.38)
8,horrible (2.69),far (-3.34)
9,rip (2.64),perfect (-3.15)


### 4.4 Regressão linear: o que explica os votos úteis? (`log(1 + votos úteis)`)

In [ ]:
Xl = pd.get_dummies(d[["tipo_produto"]], drop_first=True).astype(float)
Xl = pd.concat([Xl, d[["nota_baixa", "log_palavras", "compra_verificada", "tem_foto", "ano_num", "log_popularidade"] + list(TOPICOS)].astype(float)], axis=1)
ols = sm.OLS(np.log1p(d.helpful_vote), sm.add_constant(Xl)).fit(cov_type="HC3")
tab_ols = pd.DataFrame({"coeficiente": ols.params, "p-valor": ols.pvalues}).drop(index="const")
tab_ols["efeito aprox. (%)"] = 100 * (np.exp(tab_ols["coeficiente"]) - 1)
tab_ols = tab_ols[tab_ols["p-valor"] < 0.05].sort_values("coeficiente", ascending=False)
print(f"R² = {br(ols.rsquared, 2)} | R² ajustado = {br(ols.rsquared_adj, 2)} | n = {int(ols.nobs)}")
display(tab_ols.round(3))
RES["ols_r2"], RES["ols_r2_aj"] = float(ols.rsquared), float(ols.rsquared_adj)
RES["ols_sig"] = {k: [float(v["coeficiente"]), float(v["p-valor"]), float(v["efeito aprox. (%)"])] for k, v in tab_ols.iterrows()}

R² = 0,29 | R² ajustado = 0,29 | n = 15997


,coeficiente,p-valor,efeito aprox. (%)
nota_baixa,0.685,0.000,98.306
tem_foto,0.270,0.000,30.975
log_palavras,0.253,0.000,28.769
cobranca_cancelamento,0.181,0.000,19.898
preco_valor,0.164,0.000,17.877
atendimento,0.147,0.001,15.781
expectativa_decepcao,0.114,0.000,12.117
tipo_produto_Outros,0.095,0.016,9.997
log_popularidade,-0.054,0.000,-5.277
ano_num,-0.145,0.000,-13.481


### 4.5 Segmentação de produtos (K-means): onde agir primeiro
Só produtos com 20 avaliações ou mais na amostra.

In [ ]:
prod = d.groupby("parent_asin").agg(
    produto=("produto", "first"), tipo=("tipo_produto", "first"), n=("nota_baixa", "size"), taxa_baixa=("nota_baixa", "mean"),
    cobranca=("cobranca_cancelamento", "mean"), decepcao=("expectativa_decepcao", "mean"), qualidade=("qualidade_defeito", "mean"),
    verificada=("compra_verificada", "mean"), palavras=("n_palavras", "mean")).reset_index()
prod["n_baixa"] = (prod.n * prod.taxa_baixa).round().astype(int)
pg = prod[prod.n >= 20].copy()
FEAT_SEG = ["taxa_baixa", "cobranca", "decepcao", "qualidade", "verificada"]
Xs = StandardScaler().fit_transform(pg[FEAT_SEG].assign(log_n=np.log(pg.n)))
sil = {k: silhouette_score(Xs, KMeans(k, n_init=20, random_state=SEED).fit_predict(Xs)) for k in (3, 4, 5)}
K = max(sil, key=sil.get)
pg["segmento_id"] = KMeans(K, n_init=20, random_state=SEED).fit_predict(Xs)
seg = pg.groupby("segmento_id").agg(produtos=("parent_asin", "size"), avaliacoes=("n", "sum"), taxa_baixa=("taxa_baixa", "mean"),
                                    cobranca=("cobranca", "mean"), decepcao=("decepcao", "mean"), qualidade=("qualidade", "mean"),
                                    verificada=("verificada", "mean"), n_baixa=("n_baixa", "sum")).sort_values("taxa_baixa", ascending=False)
seg.index = [f"Segmento {chr(65 + i)}" for i in range(len(seg))]
seg["part_nota_baixa_total"] = seg.n_baixa / seg.n_baixa.sum()
print(f"Produtos com 20+ avaliações: {len(pg)} | silhueta {({k: round(v, 3) for k, v in sil.items()})} -> K = {K}")
display(seg.round(3))
seg.to_csv(PASTA_RES / "tab_segmentos.csv")
pg.sort_values("n_baixa", ascending=False).head(15)[["produto", "tipo", "n", "taxa_baixa", "n_baixa"]].to_csv(PASTA_RES / "tab_produtos_prioritarios.csv", index=False)
RES["silhueta"] = {str(k): float(v) for k, v in sil.items()}; RES["K"] = int(K); RES["n_produtos_20"] = len(pg)
RES["segmentos"] = seg.reset_index().rename(columns={"index": "segmento"}).to_dict(orient="records")

fig, (ax,) = novo_grafico(8.6, 0.8 * len(seg) + 1.8)
ax.grid(axis="x", color=GRADE, linewidth=0.8); ax.grid(axis="y", visible=False)
ordem_s = seg.sort_values("taxa_baixa")
ax.barh(range(len(ordem_s)), 100 * ordem_s.taxa_baixa, height=0.5, color=[LARANJA if v == seg.taxa_baixa.max() else AZUL for v in ordem_s.taxa_baixa])
for i, (nome, r) in enumerate(ordem_s.iterrows()):
    ax.text(100 * r.taxa_baixa + 0.8, i, f"{pct(r.taxa_baixa)}  ({int(r.produtos)} produtos · {pct(r.part_nota_baixa_total)} das notas baixas)", va="center", fontsize=8.5, color=TINTA)
ax.set_yticks(range(len(ordem_s))); ax.set_yticklabels(ordem_s.index, fontsize=9)
ax.set_xlim(0, 100 * ordem_s.taxa_baixa.max() + 42)
ax.set_xlabel("% de avaliações com nota baixa (média do segmento)", color=TINTA2)
titulo(fig, f"{K} perfis de produto: a taxa de nota baixa vai de {pct(seg.taxa_baixa.min())} a {pct(seg.taxa_baixa.max())}",
       "K-means sobre taxa de nota baixa, temas citados, compra verificada e volume (produtos com 20+ avaliações)")
fig.subplots_adjust(top=0.78, bottom=0.2, left=0.13)
salvar(fig, "g7_segmentos_produto.png")

Produtos com 20+ avaliações: 111 | silhueta {3: 0.192, 4: 0.174, 5: 0.192} -> K = 3


,produtos,avaliacoes,taxa_baixa,cobranca,decepcao,qualidade,verificada,n_baixa,part_nota_baixa_total
Segmento A,8,376,0.533,0.210,0.274,0.263,0.585,200,0.060
Segmento B,58,4101,0.327,0.125,0.228,0.206,0.872,1358,0.407
Segmento C,45,8983,0.175,0.070,0.143,0.108,0.923,1775,0.533


## Fase 5 — Avaliação
### 5.1 Limiar de decisão pelo critério de recall (escolhido no treino)
Uso de negócio do modelo B: classificar **textos de clientes sem nota** (e-mail, chat, comentários) para encaminhar
insatisfeitos ao atendimento. Atenção: o modelo foi treinado em avaliações; aplicar em outros canais exige validação.

In [ ]:
modelo_B = buscas[NOME_B][0].best_estimator_
p_oof = cross_val_predict(modelo_B, Xtr["texto_completo"], ytr, cv=cv, method="predict_proba")[:, 1]
grade = np.round(np.arange(0.05, 0.95, 0.01), 2)
rec = np.array([recall_score(ytr, p_oof >= t) for t in grade])
ok = grade[rec >= META_RECALL]
T_OP = float(ok.max()) if len(ok) else float(grade[0])
pB = probs[NOME_B]
alvo = pB >= T_OP
op = {"limiar": T_OP, "recall_treino_oof": float(recall_score(ytr, p_oof >= T_OP)), "recall_teste": float(recall_score(yte, alvo)),
      "precisao_teste": float(precision_score(yte, alvo)), "abordados_teste": int(alvo.sum()), "pct_abordados": float(alvo.mean())}
display(pd.Series(op).rename("valor").to_frame().round(3))
RES["operacao"] = op

,valor
limiar,0.340
recall_treino_oof,0.806
recall_teste,0.807
precisao_teste,0.691
abordados_teste,1406.000
pct_abordados,0.293


### 5.2 Retorno financeiro da ação (premissas na Fase 1)
Valor esperado por insatisfeito abordado = `P(cancelaria) × r × valor do assinante retido`; menos o custo do contato.
ROI = (margem incremental − investimento) ÷ investimento.

In [ ]:
ESCALA = 1000 / len(yte)


def negocio(sel, r, custo):
    margem_incr = P_CANCELA_SE_INSATISFEITO * r * VALOR_RETIDO * np.sum(yte[sel])
    invest = custo * sel.sum()
    liq = margem_incr - invest
    return {"abordados": int(sel.sum()), "margem_incremental": margem_incr, "investimento": invest, "liquido": liq,
            "retorno": margem_incr / invest if invest else np.nan, "roi": liq / invest if invest else np.nan}


linhas = []
for custo, rot_c in ((CUSTO_CONTATO, "contato ativo"), (CUSTO_COM_INCENTIVO, "contato com incentivo")):
    for r in CENARIOS_RETENCAO:
        for nome, sel in ((f"Só quem o modelo aponta (limiar {br(T_OP, 2)})", alvo), ("Todas as avaliações", np.ones(len(yte), bool))):
            n_ = negocio(sel, r, custo)
            linhas.append({"Ação": rot_c, "Retenção (r)": r, "Público": nome, "Abordados por 1.000": n_["abordados"] * ESCALA,
                           "Investimento (US$/1.000)": n_["investimento"] * ESCALA, "Margem incremental (US$/1.000)": n_["margem_incremental"] * ESCALA,
                           "Resultado líquido (US$/1.000)": n_["liquido"] * ESCALA, "Retorno (x)": n_["retorno"], "ROI": n_["roi"]})
tab_neg = pd.DataFrame(linhas)
display(tab_neg.round(2))
tab_neg.to_csv(PASTA_RES / "tab_negocio.csv", index=False)
RES["negocio"] = tab_neg.to_dict(orient="records")

,Ação,Retenção (r),Público,Abordados por 1.000,Investimento (US$/1.000),Margem incremental (US$/1.000),Resultado líquido (US$/1.000),Retorno (x),ROI
0,contato ativo,0.05,"Só quem o modelo aponta (limiar 0,34)",292.92,585.83,318.94,-266.90,0.54,-0.46
1,contato ativo,0.05,Todas as avaliações,1000.00,2000.00,395.06,-1604.94,0.20,-0.80
2,contato ativo,0.10,"Só quem o modelo aponta (limiar 0,34)",292.92,585.83,637.88,52.04,1.09,0.09
3,contato ativo,0.10,Todas as avaliações,1000.00,2000.00,790.13,-1209.88,0.40,-0.60
4,contato ativo,0.20,"Só quem o modelo aponta (limiar 0,34)",292.92,585.83,1275.75,689.92,2.18,1.18
5,contato ativo,0.20,Todas as avaliações,1000.00,2000.00,1580.25,-419.75,0.79,-0.21
6,contato com incentivo,0.05,"Só quem o modelo aponta (limiar 0,34)",292.92,1757.50,318.94,-1438.56,0.18,-0.82
7,contato com incentivo,0.05,Todas as avaliações,1000.00,6000.00,395.06,-5604.94,0.07,-0.93
8,contato com incentivo,0.10,"Só quem o modelo aponta (limiar 0,34)",292.92,1757.50,637.88,-1119.62,0.36,-0.64
9,contato com incentivo,0.10,Todas as avaliações,1000.00,6000.00,790.13,-5209.88,0.13,-0.87


In [ ]:
fig, (ax,) = novo_grafico(8.0, 4.4)
ths = np.round(np.arange(0.0, 0.96, 0.01), 2)
tons = {0.05: AZUL_CLARO, 0.10: AZUL, 0.20: AZUL_ESCURO}
melhor = {}
for r in CENARIOS_RETENCAO:
    y = np.array([negocio(pB >= t, r, CUSTO_CONTATO)["liquido"] * ESCALA for t in ths])
    ax.plot(ths, y, color=tons[r], linewidth=2, label=f"retenção r = {pct(r)}")
    melhor[r] = (float(ths[y.argmax()]), float(y.max()))
ax.axhline(0, color=EIXO, linewidth=1)
ax.axvline(T_OP, color=LARANJA, linewidth=1.4, linestyle="--")
ax.text(T_OP + 0.01, ax.get_ylim()[1] * 0.95, f"limiar escolhido\n({br(T_OP, 2)})", color=LARANJA, fontsize=8.5, va="top")
ax.set_xlim(0, 1.0)
ax.legend(frameon=False, fontsize=9, loc="lower right", labelcolor=TINTA2)
ax.set_xlabel("Limiar de probabilidade: abordamos quem tem risco igual ou maior", color=TINTA2)
ax.set_ylabel("Resultado líquido (US$ por 1.000 avaliações)", color=TINTA2)
titulo(fig, "O resultado da ação depende de quem abordamos e de quanto ela retém",
       f"Contato ativo (US$ {br(CUSTO_CONTATO, 2)} por contato), margem de {pct(MARGEM)}; r = % dos que cancelariam e são retidos")
fig.subplots_adjust(top=0.83, bottom=0.14)
salvar(fig, "g8_resultado_por_limiar.png", nota=NOTA_REAL + " | custos, margem e retenção são premissas do grupo")
RES["melhor_limiar"] = {str(k): v for k, v in melhor.items()}

### 5.3 CAC, LTV e ROI no nível do assinante (fórmulas da aula)
`LTV = margem mensal × vida média (1 ÷ cancelamento mensal)`; `LTV ÷ CAC` mostra se a aquisição se paga.
Valores de ticket, margem, churn e CAC são **premissas**.

In [ ]:
margem_mes = TICKET_MENSAL * MARGEM
linhas = []
for queda in (0.0, 0.10, 0.20):
    h = CHURN_MENSAL * (1 - queda)
    ltv = margem_mes / h
    linhas.append({"Queda relativa do cancelamento": queda, "Cancelamento mensal": h, "Vida média (meses)": 1 / h, "LTV (US$)": ltv,
                   "LTV ÷ CAC": ltv / CAC, "Ganho de LTV (US$)": ltv - margem_mes / CHURN_MENSAL})
tab_ltv = pd.DataFrame(linhas)
display(tab_ltv.round(3))
RES["ltv"] = {"margem_mes": margem_mes, "cac": CAC, "churn": CHURN_MENSAL, "tabela": tab_ltv.to_dict(orient="records")}
tab_ltv.to_csv(PASTA_RES / "tab_ltv.csv", index=False)

,Queda relativa do cancelamento,Cancelamento mensal,Vida média (meses),LTV (US$),LTV ÷ CAC,Ganho de LTV (US$)
0,0.0,0.10,10.000,105.000,2.333,0.000
1,0.1,0.09,11.111,116.667,2.593,11.667
2,0.2,0.08,12.500,131.250,2.917,26.250


### 5.4 Teste A/B: exemplo da aula e tamanho de amostra do piloto

In [ ]:
z, p_capas = proportions_ztest(np.array([100, 80]), np.array([1000, 1000]))
dif, ep = 0.02, np.sqrt(0.10 * 0.90 / 1000 + 0.08 * 0.92 / 1000)
n_capas = NormalIndPower().solve_power(proportion_effectsize(0.10, 0.08), alpha=0.05, power=0.80, alternative="two-sided")
RES["ab_capas"] = {"z": float(z), "p": float(p_capas), "ic_inf": float(dif - 1.96 * ep), "ic_sup": float(dif + 1.96 * ep), "n_por_grupo_2pp": float(n_capas)}
print(f"Exemplo da aula (capa A 80/1000 vs capa B 100/1000): z = {br(z, 2)}, p = {br(p_capas, 3)}; "
      f"IC 95% da diferença: {br(100 * (dif - 1.96 * ep), 1)} a {br(100 * (dif + 1.96 * ep), 1)} p.p.")
print(f"Para detectar 2 p.p. (8% vs 10%) com 80% de poder: {int(np.ceil(n_capas))} por grupo")

linhas = []
for r in (0.05, 0.10, 0.15, 0.20):
    p0, p1 = P_CANCELA_SE_INSATISFEITO, P_CANCELA_SE_INSATISFEITO * (1 - r)
    n_ = NormalIndPower().solve_power(proportion_effectsize(p0, p1), alpha=0.05, power=0.80, alternative="two-sided")
    linhas.append({"Redução relativa do cancelamento": r, "Cancelamento sem ação (premissa)": p0, "Cancelamento com ação": p1, "Insatisfeitos por grupo": int(np.ceil(n_))})
tab_amostra = pd.DataFrame(linhas)
display(tab_amostra.round(3))
RES["amostra_piloto"] = tab_amostra.to_dict(orient="records")
tab_amostra.to_csv(PASTA_RES / "tab_amostra_piloto.csv", index=False)

Exemplo da aula (capa A 80/1000 vs capa B 100/1000): z = 1,56, p = 0,118; IC 95% da diferença: -0,5 a 4,5 p.p.
Para detectar 2 p.p. (8% vs 10%) com 80% de poder: 3205 por grupo


,Redução relativa do cancelamento,Cancelamento sem ação (premissa),Cancelamento com ação,Insatisfeitos por grupo
0,0.05,0.5,0.475,6274
1,0.10,0.5,0.450,1565
2,0.15,0.5,0.425,693
3,0.20,0.5,0.400,388


### 5.5 Checklist de avaliação contra os critérios da Fase 1

In [ ]:
neg_base = tab_neg[(tab_neg["Ação"] == "contato ativo") & (tab_neg["Público"].str.startswith("Só"))]
r05 = neg_base[neg_base["Retenção (r)"] == 0.05].iloc[0]; r10 = neg_base[neg_base["Retenção (r)"] == 0.10].iloc[0]
n_sig_temas = int(((tab_top.p < 0.05) & (tab_top.taxa_com > base)).sum())
crit = pd.DataFrame([
    ["Temas associados a nota baixa, com teste estatístico", "≥ 1 tema com p < 0,05", f"{n_sig_temas} temas acima da média e significativos", "Sim" if n_sig_temas >= 1 else "Não"],
    ["Recall na classe 'nota baixa' (modelo B, teste)", f"≥ {pct(META_RECALL)}", pct(op["recall_teste"], 1), "Sim" if op["recall_teste"] >= META_RECALL else "Não"],
    ["Retorno da ação, cenário r = 5% (margem ÷ custo)", "> 1", f"{br(r05['Retorno (x)'], 2)}x", "Sim" if r05["Retorno (x)"] > 1 else "Não"],
    ["Retorno da ação, cenário r = 10% (margem ÷ custo)", "> 1", f"{br(r10['Retorno (x)'], 2)}x", "Sim" if r10["Retorno (x)"] > 1 else "Não"],
    ["Redução real do cancelamento", "15% relativa", "Medir em piloto", "Pendente"],
], columns=["Critério da Fase 1", "Meta", "Resultado", "Atingiu?"])
display(crit)
crit.to_csv(PASTA_RES / "tab_criterios.csv", index=False)
RES["criterios"] = crit.to_dict(orient="records")

,Critério da Fase 1,Meta,Resultado,Atingiu?
0,"Temas associados a nota baixa, com teste estatístico","≥ 1 tema com p < 0,05",6 temas acima da média e significativos,Sim
1,"Recall na classe 'nota baixa' (modelo B, teste)",≥ 80%,"80,7%",Sim
2,"Retorno da ação, cenário r = 5% (margem ÷ custo)",> 1,"0,54x",Não
3,"Retorno da ação, cenário r = 10% (margem ÷ custo)",> 1,"1,09x",Sim
4,Redução real do cancelamento,15% relativa,Medir em piloto,Pendente


## Salvando os resultados

In [ ]:
d.drop(columns=["images", "texto_completo"]).to_csv(PASTA_RES / "base_preparada.csv", index=False)


def limpar(o):
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, np.bool_):
        return bool(o)
    return str(o)


with open(PASTA_RES / "resultados.json", "w", encoding="utf-8") as f:
    json.dump(RES, f, ensure_ascii=False, indent=2, default=limpar)
print("Resultados em 'resultados/' e gráficos em 'graficos/'.")

Resultados em 'resultados/' e gráficos em 'graficos/'.
